# Parse ENTOSE National Net Generation Capacities

Notebook to parse national Net Generation Capacities provided by ENTSOE:
https://www.entsoe.eu/data/power-stats/

Use new ENTSO-E capacities from:
https://www.entsoe.eu/publications/data/power-stats/2024/net_generation_capacity_2024.csv



## Packages and options

In [1]:
import pandas as pd
import numpy as np

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
fn_in = "../source_data/net_generation_capacity_2024.csv"
dir_out = "../parsed_data/"

In [3]:
df_temp=pd.read_csv(fn_in, sep="\t")
df_temp.head()

,MeasureItem,MeasureItemCategoryID,MeasureItemID,Category,Country,Year,Representativity,ProvidedValue,CreationDate,ProvidedValueCode
0,Net Generating Capacity,6,B01,Biomass,AL,2024,100,0.0,03-03-2025 00:00:00,NaN
1,Net Generating Capacity,6,B02,Fossil Brown coal/Lignite,AL,2024,100,0.0,03-03-2025 00:00:00,NaN
2,Net Generating Capacity,6,B03,Fossil Coal-derived gas,AL,2024,100,0.0,03-03-2025 00:00:00,NaN
3,Net Generating Capacity,6,B04,Fossil Gas,AL,2024,100,0.0,03-03-2025 00:00:00,NaN
4,Net Generating Capacity,6,B05,Fossil Hard coal,AL,2024,100,0.0,03-03-2025 00:00:00,NaN


Dictionary to rename technologies names by ENTSOE to own names. Note that we only rename the lowest technology level and drop all aggregates:

In [4]:
# aggregate technologies to model level
dict_agg_tech = {       'Biomass': 'Biomass',
                        'Fossil Brown coal/Lignite': 'Lignite',
                        'Fossil Coal-derived gas': 'Gas',
                        'Fossil Gas': 'Gas',
                        'Fossil Hard coal': 'HardCoal',
                        'Fossil Oil': 'Oil',
                        'Fossil Oil shale': 'Oil',
                        'Fossil Peat': 'Lignite',
                        'Geothermal': 'Other',
                        'Hydro Pumped Storage': 'Pump',
                        'Hydro Run-of-river and poundage': 'RunOfRiver',
                        'Hydro Water Reservoir': 'Reservoir',
                        'Marine': 'Other',
                        'Nuclear': 'Nuclear',
                        'Other': 'Other',
                        'Other renewable': 'Other',
                        'Solar': 'Solar',
                        'Waste': 'Other',
                        'Wind Offshore': 'WindOffshore',
                        'Wind Onshore': 'WindOnshore'
                }

## Parse Excel File

In [6]:
df_cap = pd.read_csv(fn_in, sep="\t")
df_cap.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   MeasureItem            391 non-null    object 
 1   MeasureItemCategoryID  391 non-null    int64  
 2   MeasureItemID          391 non-null    object 
 3   Category               391 non-null    object 
 4   Country                391 non-null    object 
 5   Year                   391 non-null    int64  
 6   Representativity       391 non-null    int64  
 7   ProvidedValue          391 non-null    float64
 8   CreationDate           391 non-null    object 
 9   ProvidedValueCode      0 non-null      float64
dtypes: float64(2), int64(3), object(5)
memory usage: 30.7+ KB


Rename columns, drop aggregated columns:

In [7]:
df_cap["technology"] = df_cap.Category.map(dict_agg_tech)
df_cap = df_cap.rename(columns={'Country':'country','ProvidedValue':'MW'})
df_cap = df_cap[['country','technology','MW']].dropna()


In [8]:
df_cap.head()

,country,technology,MW
0,AL,Biomass,0.0
1,AL,Lignite,0.0
2,AL,Gas,0.0
3,AL,Gas,0.0
4,AL,HardCoal,0.0


## Aggregate technologies

In [9]:
df_agg = df_cap.copy().groupby(["country", "technology"], as_index=False).MW.sum()
df_agg.columns = ["country", "technology", "capacity"]
df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     303 non-null    object 
 1   technology  303 non-null    object 
 2   capacity    303 non-null    float64
dtypes: float64(1), object(2)
memory usage: 7.2+ KB


## Export data

In [10]:
df_agg.to_csv(dir_out + "capacities.csv", encoding="utf-8", index=False)